<a href="https://colab.research.google.com/github/yaesur/business_python/blob/%EC%B2%AD%EB%85%84%EB%A7%A4%EC%9E%85%EC%9E%84%EB%8C%80%EC%A3%BC%ED%83%9D/%EC%A0%84%EC%B2%B4_%EB%B3%80%EC%88%98%EB%8D%B0%EC%9D%B4%ED%84%B0_%EC%B6%9C%EB%A0%A5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import requests
import time

# 1. 데이터 불러오기
df = pd.read_excel('다중공선성_지역등급추가.xlsx')

# 카카오 API 키 입력 (본인의 REST API 키 입력)
from google.colab import userdata

KAKAO_API_KEY = userdata.get('KAKAO_API_KEY')

headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}

def get_subway_walk_time(address):
    try:
        # Step 1: 주소의 위경도 좌표 구하기 (지오코딩)
        url_geo = f"https://dapi.kakao.com/v2/local/search/address.json?query={address}"
        res_geo = requests.get(url_geo, headers=headers).json()

        if not res_geo['documents']:
            return None

        x = res_geo['documents'][0]['x'] # 경도
        y = res_geo['documents'][0]['y'] # 위도

        # Step 2: 해당 좌표 기준 가장 가까운 지하철역 검색 (카테고리 코드 SW8: 지하철역)
        url_subway = f"https://dapi.kakao.com/v2/local/search/category.json?category_group_code=SW8&x={x}&y={y}&radius=1000&sort=distance"
        res_subway = requests.get(url_subway, headers=headers).json()

        if not res_subway['documents']:
            return "지하철역 없음(1km 이내)"

        # 가장 가까운 지하철역과의 직선 거리(m)
        distance_m = float(res_subway['documents'][0]['distance'])

        # Step 3: 직선 거리를 기반으로 대략적인 도보 시간 계산
        # (성인 평균 도보 속도: 분당 약 67m ~ 80m 계산, 여기서는 대략 분당 75m 기준)
        # ※ 카카오 길찾기(도보) API를 사용하면 완벽한 실도보 시간이 나오나,
        #   로컬 카테고리 API의 직선거리 정보로 추정하는 것이 API 요청 제한을 피하기에 효율적입니다.
        walk_time = round(distance_m / 75)

        return f"{walk_time}분"

    except Exception as e:
        return None

# 2. 반복문을 통해 '교통등급' 컬럼에 도보 시간 채우기
# API 과부하 방지를 위해 주소지가 있는 행만 처리하거나, 데이터가 많다면 time.sleep()을 활용합니다.
for idx, row in df.iterrows():
    if pd.notna(row['주소지']):
        # 주소지에 괄호가 포함되어 있다면(예: 백년빌) API 인식을 위해 정제해주는 것이 좋습니다.
        clean_address = row['주소지'].split('(')[0].strip()

        walk_time = get_subway_walk_time(clean_address)
        df.at[idx, '교통등급'] = walk_time

        # API 호출 간격 조절 (초당 요청 제한 방지)
        time.sleep(0.1)

# 3. 결과 확인 및 저장
print(df[['주택명', '주소지', '교통등급']].head())
df.to_csv('다중공선성_교통등급완료.csv', index=False, encoding='utf-8-sig')

/tmp/ipykernel_1847/304225106.py:56: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '13분' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, '교통등급'] = walk_time


          주택명                         주소지 교통등급
0         백년빌               강남구 개포로24길 13  13분
1     강동서영스윗홈             강동구 천중로48길 9-13   5분
2     강동서영스윗홈             강동구 천중로48길 9-13   5분
3  수유동 569-17  강북구 삼양로131길 68, 수유동 569-17   5분
4     자양에스하임4               광진구 뚝섬로49길 60   8분


In [8]:
import re
import requests
import time
import pandas as pd
from google.colab import userdata

print("🚀 '지하철역 없음' 데이터 대상 [20km 반경 확장 탐색]을 시작합니다...")

# 1. 런타임이 끊겼을 때를 대비해 headers와 API Key를 이 셀 안에서 확실하게 다시 정의합니다.
KAKAO_API_KEY = userdata.get('KAKAO_API_KEY')
headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY.strip()}"}

# 2. 이전 셀의 df 데이터를 그대로 이어받아 반복문을 돕니다.
for idx, row in df.iterrows():
    current_grade = str(row.get('교통등급', '')).strip()

    # ⚠️ 오직 '지하철역 없음' 텍스트가 포함된 데이터만 골라냅니다.
    if '지하철역 없음' not in current_grade:
        continue

    if pd.notna(row['주소지']):
        address_str = str(row['주소지']).strip()

        # 주소 형식 깔끔하게 정제 (괄호 제거 및 콤마 뒤 텍스트 제거)
        clean_address = re.sub(r'\([^)]*\)', '', address_str).strip()
        if ',' in clean_address:
            clean_address = clean_address.split(',')[0].strip()

        if not (clean_address.startswith('서울') or clean_address.startswith('수도권')):
            clean_address = "서울특별시 " + clean_address
        clean_address = " ".join(clean_address.split())

        try:
            # Step 1: 주소의 위경도 좌표 구하기 (지오코딩)
            url_geo = f"https://dapi.kakao.com/v2/local/search/address.json?query={clean_address}"
            res_geo = requests.get(url_geo, headers=headers).json()

            if not res_geo.get('documents'):
                continue

            x = res_geo['documents'][0]['x']  # 경도
            y = res_geo['documents'][0]['y']  # 위도

            # Step 2: radius를 최대치인 20000(20km)으로 늘려 무조건 탐색합니다.
            url_subway = f"https://dapi.kakao.com/v2/local/search/category.json?category_group_code=SW8&x={x}&y={y}&radius=20000&sort=distance"
            res_subway = requests.get(url_subway, headers=headers).json()

            if not res_subway.get('documents'):
                print(f"[{idx+1}] 주택명: {row['주택명']} ➔ 20km 반경 내에도 역이 없습니다.")
                continue

            # 가장 가까운 지하철역 이름과 직선 거리(m) 추출
            distance_m = float(res_subway['documents'][0]['distance'])
            station_name = res_subway['documents'][0]['place_name']

            # Step 3: 도보 시간 환산 (실도로 굴곡 보정 1.35배 반영 후 분당 70m 기준 연산)
            real_walk_distance = distance_m * 1.35
            walk_time = round(real_walk_distance / 70)
            result_time = f"{max(1, walk_time)}분"

            # Step 4: 기존 '지하철역 없음' 칸에 실제 시간 데이터 업데이트
            df.at[idx, '교통등급'] = result_time

            print(f"[{idx+1}/{len(df)}] [보충 완료] 주택명: {row['주택명']} ➔ 가장 가까운 역: {station_name} (거리: {round(distance_m)}m) ➔ 결과: {result_time}")

            # API 요청 제한 방지 딜레이
            time.sleep(0.08)

        except Exception as e:
            print(f"[{idx+1}] 처리 중 에러 발생: {e}")
            continue

print("\n✨ '지하철역 없음' 데이터 보충이 완전히 종료되었습니다!")

# 최종 결과 저장
df.to_csv('다중공선성_교통등급완료.csv', index=False, encoding='utf-8-sig')

🚀 '지하철역 없음' 데이터 대상 [20km 반경 확장 탐색]을 시작합니다...
[9/1132] [보충 완료] 주택명: 아임빌 7차 ➔ 가장 가까운 역: 금천구청역 1호선 (거리: 1286m) ➔ 결과: 25분
[13/1132] [보충 완료] 주택명: 휴먼에코빌 4차 ➔ 가장 가까운 역: 구로디지털단지역 2호선 (거리: 1154m) ➔ 결과: 22분
[14/1132] [보충 완료] 주택명: 휴먼에코빌 4차 ➔ 가장 가까운 역: 구로디지털단지역 2호선 (거리: 1154m) ➔ 결과: 22분
[15/1132] [보충 완료] 주택명: 씨앤제이빌 ➔ 가장 가까운 역: 방학역 1호선 (거리: 1065m) ➔ 결과: 21분
[17/1132] [보충 완료] 주택명: 삼화에코빌2차 ➔ 가장 가까운 역: 용마산역 7호선 (거리: 1526m) ➔ 결과: 29분
[22/1132] [보충 완료] 주택명: 홍은동 411-5 ➔ 가장 가까운 역: 가좌역 경의중앙선 (거리: 1564m) ➔ 결과: 30분
[23/1132] [보충 완료] 주택명: 르비앙휴 ➔ 가장 가까운 역: 가좌역 경의중앙선 (거리: 1611m) ➔ 결과: 31분
[24/1132] [보충 완료] 주택명: 르비앙휴 ➔ 가장 가까운 역: 홍제역 3호선 (거리: 1241m) ➔ 결과: 24분
[25/1132] [보충 완료] 주택명: 현인그린빌 ➔ 가장 가까운 역: 녹번역 3호선 (거리: 1103m) ➔ 결과: 21분
[63/1132] [보충 완료] 주택명: 낙원가우디움 ➔ 가장 가까운 역: 오류동역 1호선 (거리: 1311m) ➔ 결과: 25분
[64/1132] [보충 완료] 주택명: 낙원가우디움 ➔ 가장 가까운 역: 오류동역 1호선 (거리: 1311m) ➔ 결과: 25분
[70/1132] [보충 완료] 주택명: 아이빌10차 ➔ 가장 가까운 역: 금천구청역 1호선 (거리: 1561m) ➔ 결과: 30분
[71/1132] [보충 완료] 주택명: 아이빌10차 ➔ 가장 가까운 역: 금천구청역 1호선 (거리: 1561m) ➔ 결과:

In [10]:
import re
import pandas as pd

print("📊 전수 데이터 분석 기반 자치구별 공급밀도 연산을 시작합니다...")

# [참고] 서울시 25개 자치구 전체 최신 청년 인구수 (20~39세 통계 통제 변수)
seoul_youth_population = {
    '종로구': 41000, '중구': 37000, '용산구': 67000, '성동구': 88000, '광진구': 114000,
    '동대문구': 109000, '중랑구': 105000, '성북구': 137500, '강북구': 79000, '도봉구': 78000,
    '노원구': 134000, '은평구': 129000, '서대문구': 97000, '마포구': 123831, '양천구': 113000,
    '강서구': 177500, '구로구': 115000, '금천구': 78000, '영등포구': 125000, '동작구': 122000,
    '관악구': 195000, '서초구': 114000, '강남구': 152000, '송파구': 193000, '강동구': 99820
}

df1 = pd.read_csv('다중공선성_교통등급완료 (1).csv')

# Step 1: 현재 데이터프레임(df) 내에서 각 자치구별 '총 모집 건수(행 개수)'를 계산합니다.
# 데이터에 '자치구' 컬럼이 이미 깨끗하게 있으므로 이를 활용합니다.
district_counts = df1['자치구'].value_counts().to_dict()

print("\n🔍 [분석 결과] 자치구별 전수 데이터 공급 건수:")
for gu, count in district_counts.items():
    print(f" - {gu}: {count}건 공급됨")

# Step 2: 연산 및 매핑 함수 정의
def get_full_data_density(row):
    gu = row['자치구']

    # 데이터프레임 내부에서 구한 이 구의 총 공급 건수
    total_supply = district_counts.get(gu, 0)

    # 이 구의 청년 인구수
    youth_pop = seoul_youth_population.get(gu, None)

    if youth_pop and total_supply > 0:
        # 공급 밀도 = 총 공급 건수 / 청년 인구수
        density = total_supply / youth_pop
        return pd.Series([total_supply, density])

    return pd.Series([total_supply, None])

# Step 3: '모집규모밀도' 컬럼과 인구대비 밀도 데이터 덮어씌우기
# 데이터셋에 있는 기존 컬럼명인 '모집규모밀도'에 바로 값을 주입합니다.
df1[['모집규모밀도_건수', '모집규모밀도']] = df1.apply(get_full_data_density, axis=1)

# 보기 편하게 기존 변수 위치 정제 (선택사항)
if '모집규모밀도_건수' in df1.columns:
    print("\n💡 '모집규모밀도' 컬럼에 [총 공급건수 / 청년인구] 연산 값이 성공적으로 대입되었습니다.")

print("\n✨ 최종 매핑 샘플 확인 (상위 5개):")
print(df1[['자치구', '주택명', '모집규모밀도_건수', '모집규모밀도']].head())

# 4. 결과 저장
df1.to_csv('다중공선성_교통_공급밀도최종완료.csv', index=False, encoding='utf-8-sig')

📊 전수 데이터 분석 기반 자치구별 공급밀도 연산을 시작합니다...

🔍 [분석 결과] 자치구별 전수 데이터 공급 건수:
 - 강동구: 144건 공급됨
 - 구로구: 122건 공급됨
 - 송파구: 115건 공급됨
 - 영등포구: 83건 공급됨
 - 금천구: 78건 공급됨
 - 강북구: 67건 공급됨
 - 관악구: 66건 공급됨
 - 광진구: 49건 공급됨
 - 성북구: 45건 공급됨
 - 중랑구: 44건 공급됨
 - 양천구: 40건 공급됨
 - 서초구: 39건 공급됨
 - 은평구: 37건 공급됨
 - 서대문구: 37건 공급됨
 - 동대문구: 31건 공급됨
 - 강남구: 28건 공급됨
 - 마포구: 28건 공급됨
 - 강서구: 15건 공급됨
 - 성동구: 14건 공급됨
 - 도봉구: 13건 공급됨
 - 동작구: 13건 공급됨
 - 중구: 11건 공급됨
 - 노원구: 7건 공급됨
 - 종로구: 6건 공급됨

💡 '모집규모밀도' 컬럼에 [총 공급건수 / 청년인구] 연산 값이 성공적으로 대입되었습니다.

✨ 최종 매핑 샘플 확인 (상위 5개):
   자치구         주택명  모집규모밀도_건수    모집규모밀도
0  강남구         백년빌       28.0  0.000184
1  강동구     강동서영스윗홈      144.0  0.001443
2  강동구     강동서영스윗홈      144.0  0.001443
3  강북구  수유동 569-17       67.0  0.000848
4  광진구     자양에스하임4       49.0  0.000430
